In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.tools import TavilySearchResults 
from langchain.agents import AgentType, initialize_agent, Tool
from langchain_openai import AzureChatOpenAI
import os

import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

In [ ]:
os.environ["TAVILY_API_KEY"] = "tavily_key"

In [ ]:
## PDF Document Indexing with FAISS This script loads PDF files from a given folder, extracts text from them, and creates a FAISS index using sentence-transformer embeddings. FAISS enables efficient similarity search, making it useful for retrieval-based applications.  

def load_and_index_docs(docs_folder: str) -> FAISS:
    """
    Load PDF documents from a folder and create a FAISS index
    """
    documents = []
    for file in os.listdir(docs_folder):
        if file.endswith('.pdf'):
            loader = PyPDFLoader(os.path.join(docs_folder, file))
            documents.extend(loader.load())

    print(f"Loaded {len(documents)} pages")
    
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return FAISS.from_documents(documents, embeddings)
## Retrieval-Augmented Generation (RAG) Agent This function creates a RAG agent that combines a vector database for document retrieval and web search for external knowledge. The agent follows a ReAct (Reasoning + Acting) framework to determine when to query stored documents or perform an internet search.  

In [ ]:
def create_rag_agent(docs_folder: str, llm: AzureChatOpenAI):
    # Create vector store
    vectorstore = load_and_index_docs(docs_folder)
    
    # Define tools
    tools = [
        Tool(
            name="Knowledge Base",
            func=lambda q: vectorstore.similarity_search(q, k=2),
            description="Useful for searching information from the knowledge base documents. Returns relevant document chunks."
        ),
        Tool(
            name="Web Search",
            func=TavilySearchResults().run,
            description="Useful for searching current information from the internet when knowledge base doesn't have the answer."
        )
    ]
      # Initialize ReAct agent
    agent = initialize_agent(
        tools=tools,
        llm=llm,
        agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
        verbose=True,
        handle_parsing_errors=True,
        system_message="""You are a helpful AI assistant with access to two sources of information:
                1. A knowledge base of laptop manuals consisting of techincal guides and troubleshooting.
                    To answer any queries about laptops, you can search the knowledge base.
                2. Web search capability (use the Web Search tool)

                For each user query:
                - First try to find relevant information in the knowledge base
                - If no relevant information is found, perform a web search
                - Be conversational and friendly
                - Respond directly to greetings without using any tools
                - Cite your sources when providing information

                Keep responses clear and concise while maintaining a helpful tone."""
            )
    return agent

In [ ]:
## Initialise the LLM
llm = AzureChatOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    temperature=0.2, top_p=1
)

In [ ]:
rag_agent = create_rag_agent("data/manuals", llm)
## Chat Interface for the RAG Agent This function facilitates conversations with the RAG agent, maintaining chat history for context-aware responses.  

chat_history = []

def chat_with_agent(query: str):
    global chat_history
    response = rag_agent.invoke({
        "input": query,
        "chat_history": chat_history
    })
    chat_history.append({"role": "user", "content": response["input"]})
    chat_history.append({"role": "assistant", "content": response["output"]})
    return response

In [ ]:
response = chat_with_agent("How to fix a laptop that won't turn on?")
response
response = chat_with_agent("How to install a new operating system on a laptop?")
response
response = chat_with_agent("How to fix a laptop that is overheating?")
response['output']